In [ ]:
import torch
from pathlib import Path
import torch.nn.functional as F
import h5py
import matplotlib.pyplot as plt
import loss
import pandas as pd
import json
import numpy as np
import stemdiff as sd

dataset = h5py.File("dataset_filtered/train_target.h5", 'r')

In [ ]:

with open("dataset_filtered/dataset_params.json", "r") as f:
    center_sizes = json.load(f)["center_sizes"]

In [ ]:
def resize_profile(q, I, calibration_constant):
    x = np.arange(int(q[-1] * calibration_constant) + 1)
    y = np.interp(x, q * calibration_constant, I, left=0, right=0)
    return y

# Calculate constants

In [ ]:
shifts = {
    # "au": 1.01,
    "feo": 0.99,
    "gdf3": 0.95,
    "laf3": 0.98,
    "tbf3": 0.94
}
scale_factor = 4
calibration_constants = {}
for n, d in dataset.items():
    print(n)
    print(len(d))
    input2d = torch.from_numpy(d[:])[:, None].float()
    df = pd.read_csv(Path("dataset_filtered") / n , sep=r'\s+')
    db = sd.dbase.read_database(Path("dataset_filtered") / "dbase" / f"db_train_{n}")
    centers = db[["Xcenter", "Ycenter"]].to_numpy() / 4
    target = (df.q.to_numpy(), df.I.to_numpy(), center_sizes[n] * scale_factor)
    rad_dist = loss.RadialDistribution(256 * scale_factor, 256 * scale_factor, input2d.device)
    with torch.no_grad():
        device = input2d.device
        summed_input2d = loss.center_images(input2d, centers).sum(dim=0)
        if scale_factor != 1:
            summed_input2d = F.interpolate(summed_input2d.unsqueeze(0), scale_factor=scale_factor, mode="bicubic")
        radial_distance, intensity = rad_dist(summed_input2d.squeeze())

        q, I, center_size = target
        q, I = torch.from_numpy(q), torch.from_numpy(I)

        # remove center and normalize
        intensity[:center_size] = 0
        intensity = intensity / intensity[:intensity.shape[0] // 2].max()

        max_target = q[I.argmax()]
        max_input = torch.argmax(intensity[:intensity.shape[0] // 2])
        # calibration_constant = max_xrd / max_eld, this is pixels -> q
        # we are going q -> pixels, so it needs to be max_eld / max_xrd
        calibration_constant = max_input / max_target
        calibration_constant = calibration_constant / shifts.get(n, 1)
        # target1d = torch.from_numpy(resize_profile(q, I, calibration_constant ))
        target1d = loss.resize_target(q, I, calibration_constant, nearest=False)

        # Save constant for original resolution
        calibration_constants[n] = calibration_constant.item() / scale_factor

        target1d = torch.nn.functional.pad(target1d, (0, intensity.shape[0] - target1d.shape[0]))
    
    # plt.imshow(torch.log10_(summed_input2d[0] + 1).cpu())
    # plt.show()
    plt.plot(target1d[:100 * scale_factor])
    plt.plot(intensity[:100 * scale_factor])
    plt.show()

    # plt.figure()
    # plt.plot(resize_profile(q, I, calibration_constant )[:100])
    # plt.plot(intensity[:100])
    # plt.show()

    # plt.figure()
    # plt.plot(q, I)
    # plt.show()

    del input2d

print(json.dumps(calibration_constants, indent=4))

# Verify different scale and save target profiles

In [ ]:
target_profiles = {}

In [ ]:
scale_factor = 1
for n, d in dataset.items():
    print(n)
    print(len(d))
    input2d = torch.from_numpy(d[:])[:, None].float()
    df = pd.read_csv(Path("dataset_filtered") / n , sep=r'\s+')
    db = sd.dbase.read_database(Path("dataset_filtered") / "dbase" / f"db_train_{n}")
    centers = db[["Xcenter", "Ycenter"]].to_numpy() / 4
    target = (df.q.to_numpy(), df.I.to_numpy(), center_sizes[n] * scale_factor)
    rad_dist = loss.RadialDistribution(256 * scale_factor, 256 * scale_factor, input2d.device)
    with torch.no_grad():
        device = input2d.device
        summed_input2d = loss.center_images(input2d, centers).sum(dim=0)
        if scale_factor != 1:
            summed_input2d = F.interpolate(summed_input2d.unsqueeze(0), scale_factor=scale_factor, mode="bicubic")
        summed_input2d = summed_input2d.clamp(0, None)
        radial_distance, intensity = rad_dist(summed_input2d.squeeze())

        q, I, center_size = target
        q, I = torch.from_numpy(q), torch.from_numpy(I)

        # remove center and normalize
        intensity[:center_size] = 0
        intensity = intensity / intensity[:intensity.shape[0] // 2].max()

        calibration_constant = calibration_constants[n] * scale_factor
        target1d = loss.resize_target(q, I, calibration_constant, nearest=False)

        # Save target profiles
        if scale_factor == 1:
            target_profiles[n] = target1d
        else:
            target_profiles[f"{n}x{scale_factor}"] = target1d

        target1d = torch.nn.functional.pad(target1d, (0, intensity.shape[0] - target1d.shape[0]))
    
    # plt.imshow(torch.log10_(summed_input2d[0, 0] + 1).cpu())
    # plt.show()
    plt.plot(target1d[:100 * scale_factor])
    plt.plot(intensity[:100 * scale_factor])
    plt.show()

    del input2d

## Save profile files

In [ ]:
# Show saved profiles
target_profiles.keys()

In [ ]:
for n, p in target_profiles.items():
    np.savetxt(f"dataset_filtered/target_profiles/{n}", p)